In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    classification_report
)

import streamlit as st

import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


In [2]:
matches = pd.read_csv(r"D:\personal project\results.csv")

rankings = pd.read_csv(r"D:\personal project\fifa_rankings.csv")

fixtures = pd.read_csv(r"D:\personal project\wc2026_fixtures.csv")

In [3]:
print(matches.columns)

print(rankings.columns)

print(fixtures.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='object')
Index(['Unnamed: 0', 'rank', 'country_full', 'country_abrv', 'total_points',
       'previous_points', 'rank_change', 'confederation', 'rank_date'],
      dtype='object')
Index(['group', 'matchday', 'date', 'team1', 'team2'], dtype='object')


In [4]:


matches["date"] = pd.to_datetime( matches["date"])

rankings["rank_date"] = pd.to_datetime(rankings["rank_date"])

fixtures["date"] = pd.to_datetime(fixtures["date"])

print("Dates converted successfully.")

Dates converted successfully.


In [5]:
rankings["country_full"] = rankings[
    "country_full"
].replace({

    # USA naming difference
    "USA": "United States",

    # Iran naming difference
    "IR Iran": "Iran",

    # South Korea naming difference
    "Korea Republic": "South Korea",

    # Ivory Coast naming difference
    "Côte d'Ivoire": "Ivory Coast"

})

print("Country names fixed successfully.")

Country names fixed successfully.


In [6]:
matches = matches.sort_values("date")

rankings = rankings.sort_values("rank_date")

print("Datasets sorted successfully.")

Datasets sorted successfully.


In [ ]:
# CREATE HOME TEAM RANKINGS

home_rankings = rankings[[
    "country_full",
    "rank",
    "total_points",
    "rank_date"
]].copy()

home_rankings = home_rankings.rename(columns={

    "country_full": "home_team",

    "rank": "home_rank",

    "total_points": "home_points"

})

# SORT AGAIN


home_rankings = home_rankings.sort_values(
    "rank_date"
)

# MERGE HOME TEAM RANKINGS

merged = pd.merge_asof(

    matches.sort_values("date"),

    home_rankings,

    left_on="date",

    right_on="rank_date",

    left_by="home_team",

    right_by="home_team",

    direction="backward"

)

print(merged.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'home_rank', 'home_points',
       'rank_date'],
      dtype='object')


In [8]:
print(merged.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'home_rank', 'home_points',
       'rank_date'],
      dtype='object')


In [ ]:
# CREATE AWAY TEAM RANKINGS

away_rankings = rankings[[
    "country_full",
    "rank",
    "total_points",
    "rank_date"
]].copy()

away_rankings = away_rankings.rename(columns={

    "country_full": "away_team",

    "rank": "away_rank",

    "total_points": "away_points"

})

# ============================================
# SORT AGAIN
# ============================================

away_rankings = away_rankings.sort_values(
    "rank_date"
)

# ============================================
# MERGE AWAY TEAM RANKINGS
# ============================================

merged = pd.merge_asof(

    merged.sort_values("date"),

    away_rankings,

    left_on="date",

    right_on="rank_date",

    left_by="away_team",

    right_by="away_team",

    direction="backward"

)

print(merged.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'home_rank', 'home_points',
       'rank_date_x', 'away_rank', 'away_points', 'rank_date_y'],
      dtype='object')


In [10]:
print(merged.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'home_rank', 'home_points',
       'rank_date_x', 'away_rank', 'away_points', 'rank_date_y'],
      dtype='object')


In [ ]:
columns_to_drop = [

    "rank_date_x",

    "rank_date_y"

]

for col in columns_to_drop:

    if col in merged.columns:

        merged = merged.drop(
            columns=[col]
        )

merged = merged.dropna()

print("Dataset cleaned successfully.")

print(merged.head())

print("\nDataset Shape:")

print(merged.shape)

Dataset cleaned successfully.
            date     home_team     away_team  home_score  away_score  \
18704 1993-01-01         Ghana          Mali         1.0         1.0   
18705 1993-01-02         Gabon  Burkina Faso         1.0         1.0   
18706 1993-01-02        Kuwait       Lebanon         2.0         0.0   
18707 1993-01-03  Burkina Faso          Mali         1.0         0.0   
18708 1993-01-03         Gabon         Ghana         2.0         3.0   

      tournament         city country  neutral  home_rank  home_points  \
18704   Friendly   Libreville   Gabon     True       39.0         34.0   
18705   Friendly   Libreville   Gabon    False       55.0         27.0   
18706   Friendly  Kuwait City  Kuwait    False       71.0         21.0   
18707   Friendly   Libreville   Gabon     True       97.0         11.0   
18708   Friendly   Libreville   Gabon    False       55.0         27.0   

       away_rank  away_points  
18704       69.0         22.0  
18705       97.0         11.

In [ ]:
# MODERN FOOTBALL ONLY

merged = merged[
    merged["date"] >= "2010-01-01"
]

print(merged.shape)

print(merged.head())

(13138, 13)
            date home_team    away_team  home_score  away_score tournament  \
33584 2010-01-02     Qatar         Mali         0.0         0.0   Friendly   
33585 2010-01-02     Syria     Zimbabwe         6.0         0.0   Friendly   
33586 2010-01-02     Yemen   Tajikistan         0.0         1.0   Friendly   
33588 2010-01-04     Egypt         Mali         1.0         0.0   Friendly   
33589 2010-01-04  Tanzania  Ivory Coast         0.0         1.0   Friendly   

                city               country  neutral  home_rank  home_points  \
33584           Doha                 Qatar    False       86.0        376.0   
33585   Kuala Lumpur              Malaysia     True       91.0        347.0   
33586         Sana'a                 Yemen    False      130.0        176.0   
33588          Dubai  United Arab Emirates     True       24.0        832.0   
33589  Dar es Salaam              Tanzania    False      106.0        290.0   

       away_rank  away_points  
33584       

In [13]:

# Rank Difference
merged["rank_difference"] = (
 merged["home_rank"] - merged["away_rank"]
)

# FIFA POINTS DIFFERENCE

merged["points_difference"] = (
     merged["home_points"]- merged["away_points"]
)

# GOAL DIFFERENCE

merged["goal_difference"] = (
merged["home_score"] -merged["away_score"]
)

# TOTAL GOALS

merged["total_goals"] = (
merged["home_score"]+ merged["away_score"]
)


# MATCH RESULT
# 0 = HOME WIN
# 1 = AWAY WIN
# 2 = DRAW


def get_result(row):

    if row["home_score"] > row["away_score"]:

        return 0

    elif row["home_score"] < row["away_score"]:

        return 1

    else:

        return 2

merged["result"] = merged.apply(
    get_result,
    axis=1
)

print("Feature engineering completed.")

print(merged.head())

Feature engineering completed.
            date home_team    away_team  home_score  away_score tournament  \
33584 2010-01-02     Qatar         Mali         0.0         0.0   Friendly   
33585 2010-01-02     Syria     Zimbabwe         6.0         0.0   Friendly   
33586 2010-01-02     Yemen   Tajikistan         0.0         1.0   Friendly   
33588 2010-01-04     Egypt         Mali         1.0         0.0   Friendly   
33589 2010-01-04  Tanzania  Ivory Coast         0.0         1.0   Friendly   

                city               country  neutral  home_rank  home_points  \
33584           Doha                 Qatar    False       86.0        376.0   
33585   Kuala Lumpur              Malaysia     True       91.0        347.0   
33586         Sana'a                 Yemen    False      130.0        176.0   
33588          Dubai  United Arab Emirates     True       24.0        832.0   
33589  Dar es Salaam              Tanzania    False      106.0        290.0   

       away_rank  away_po

In [ ]:

# RANDOM FOREST CLASSIFIER
# MATCH RESULT PREDICTION

# FEATURES
features = [
"rank_difference",
"points_difference"
]

X = merged[features]

# TARGET
# 0 = HOME WIN
# 1 = AWAY WIN
# 2 = DRAW

y = merged["result"]

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# CREATE CLASSIFIER

classifier = RandomForestClassifier(n_estimators=200, random_state=42)

# TRAIN MODEL
classifier.fit(X_train, y_train)

print("Classifier trained successfully.")

# TEST MODEL

predictions = classifier.predict(X_test)

# ACCURACY

accuracy = accuracy_score(y_test,predictions)

print("\nAccuracy:")

print(accuracy)

Classifier trained successfully.

Accuracy:
0.4984779299847793


In [15]:
features = [
    "rank_difference",

    "points_difference"
]
X= merged[features]
y_home=merged["home_score"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y_home, test_size=0.2, random_state=42)

In [ ]:
home_goal_model = RandomForestRegressor(n_estimators=200,random_state=42)

# TRAIN MODEL
home_goal_model.fit(X_train,y_train)

print("Home goal prediction model trained.")

# TEST MODEL

predictions = home_goal_model.predict(X_test)

# EVALUATE MODEL

mae = mean_absolute_error(y_test,predictions)

print("\nMean Absolute Error:")

print(mae)

Home goal prediction model trained.

Mean Absolute Error:
1.177914447116275


In [ ]:
features = [
"rank_difference",
 "points_difference"]

X = merged[features]

# AWAY GOALS TARGET

y_away = merged["away_score"]

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(X,y_away,test_size=0.2, random_state=42)

In [ ]:
away_goal_model = RandomForestRegressor(n_estimators=200,random_state=42)

# TRAIN MODEL

away_goal_model.fit(X_train,y_train)

print("Away goal prediction model trained.")

# TEST MODEL

predictions = away_goal_model.predict(X_test)

# EVALUATE MODEL

mae = mean_absolute_error(y_test, predictions)

print("\nMean Absolute Error:")

print(mae)

Away goal prediction model trained.

Mean Absolute Error:
0.929231644849949


In [20]:

# TEAM RANKINGS DICTIONARY
latest_rankings = rankings.sort_values(
    "rank_date"
).drop_duplicates(
    "country_full",
    keep="last"
)

team_rankings = {}

for _, row in latest_rankings.iterrows():

    team_rankings[row["country_full"]] = {

        "rank": row["rank"],

        "points": row["total_points"]

    }

print("Team rankings dictionary created.")

Team rankings dictionary created.


In [21]:

# MATCH PREDICTION FUNCTION

def predict_match(team1, team2):


    # GET TEAM RANKINGS


    team1_rank = team_rankings[team1]["rank"]

    team2_rank = team_rankings[team2]["rank"]

    team1_points = team_rankings[team1]["points"]

    team2_points = team_rankings[team2]["points"]


    # CREATE FEATURES


    rank_difference = ( team1_rank - team2_rank)

    points_difference = (team1_points - team2_points)

    X = pd.DataFrame([{
        "rank_difference": rank_difference,

        "points_difference": points_difference

    }])

    # PREDICT WIN PROBABILITIES


    probabilities = classifier.predict_proba(X)[0]

   
    # PREDICT SCORES


    home_goals = round(

        home_goal_model.predict(X)[0]

    )

    away_goals = round(

        away_goal_model.predict(X)[0]

    )


    # PRINT RESULTS


    print(f"\n{team1} {home_goals} - {away_goals} {team2}")

    print("\nWin Probabilities:\n")

    print(f"{team1}: {probabilities[0]*100:.2f}%")

    print(f"{team2}: {probabilities[1]*100:.2f}%")

    print(f"Draw: {probabilities[2]*100:.2f}%")

In [22]:
predict_match("Brazil", "France")


Brazil 1 - 2 France

Win Probabilities:

Brazil: 6.00%
France: 72.50%
Draw: 21.50%


In [23]:
print(classifier.classes_)


[0 1 2]


In [24]:
import joblib

joblib.dump(

    classifier,

    "classifier.pkl"

)

print("Classifier saved.")

Classifier saved.


In [25]:
joblib.dump(

    home_goal_model,

    "home_goal_model.pkl"

)

print("Home goal model saved.")

Home goal model saved.


In [26]:
joblib.dump(

    away_goal_model,

    "away_goal_model.pkl"

)

print("Away goal model saved.")

Away goal model saved.
